# Download Arsenal

Ноутбук использует две готовые конфигурации: многомодельную `test_playbook_arsenal_router_mode.toml` для Router Mode и `test_playbook_arsenal_model_mode.toml` с одной моделью на сервер для Model Mode. Уже загруженные файлы повторно не скачиваются.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = next(
    directory
    for directory in (Path.cwd(), *Path.cwd().parents)
    if (directory / ".zemicomp").is_file() and (directory / "zemi").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from zemi.playbook import arsenal

## Загрузка

Следующие ячейки выполняют реальные сетевые загрузки и могут работать продолжительное время. Прогресс каждого llama-сервера и каждой модели отображается в output ячейки.

In [ ]:
router_mode_arsenal = arsenal.download(
    "@comp/tests/playbook_arsenal/test_playbook_arsenal_router_mode.toml"
)

In [ ]:
model_mode_arsenal = arsenal.download(
    "@comp/tests/playbook_arsenal/test_playbook_arsenal_model_mode.toml"
)

# Begin и end playbook

Ниже приведены осмысленные варианты запуска и завершения. Каждая ячейка запускает или останавливает реальные процессы `llama-server`. Не запускайте несколько сценариев `begin_playbook` подряд без промежуточного `end_playbook(stop_arsenal_after_end=True)`, если в новом сценарии не включена предварительная остановка.

### Model Mode: чистый запуск

Рекомендуемый вариант для воспроизводимого запуска: сначала остановить возможные старые процессы Arsenal, затем запустить по одному серверу на модель и остановить их после playbook.

In [ ]:
model_mode_arsenal.begin_playbook(
    stop_arsenal_before_begin=True,
    llama_router_mode=False,
)

# Здесь выполняются шаги playbook.

model_mode_arsenal.end_playbook(stop_arsenal_after_end=True)

### Model Mode: запуск без предварительной остановки

Используйте только когда известно, что настроенные порты свободны. Завершение с `False` намеренно оставляет серверы работающими для следующего playbook; последняя строка показывает явную последующую очистку.

In [ ]:
model_mode_arsenal.begin_playbook(
    stop_arsenal_before_begin=False,
    llama_router_mode=False,
)

# Серверы остаются доступны после завершения playbook.
model_mode_arsenal.end_playbook(stop_arsenal_after_end=False)

# Выполните позже, когда серверы больше не нужны.
model_mode_arsenal.end_playbook(stop_arsenal_after_end=True)

## Router Mode: чистый запуск

Каждый сервер получает все свои модели через сгенерированный INI-пресет. Предварительная и завершающая остановка делают сценарий полностью изолированным.

In [ ]:
router_mode_arsenal.begin_playbook(
    stop_arsenal_before_begin=True,
    llama_router_mode=True,
)

# Здесь выполняются шаги playbook с выбором модели в запросах.

router_mode_arsenal.end_playbook(stop_arsenal_after_end=True)

## Router Mode: запуск на заведомо свободных портах

Вариант без предварительной остановки полезен, когда состояние окружения контролируется снаружи. Серверы останавливаются после playbook.

In [ ]:
router_mode_arsenal.begin_playbook(
    stop_arsenal_before_begin=False,
    llama_router_mode=True,
)

# Здесь выполняются шаги playbook с выбором модели в запросах.

router_mode_arsenal.end_playbook(stop_arsenal_after_end=True)

## Проверка ограничения Model Mode

Исходный Arsenal содержит несколько моделей на сервер. Поэтому попытка запустить его без Router Mode ожидаемо завершается `ValueError` до запуска первого сервера.

In [ ]:
try:
    router_mode_arsenal.begin_playbook(
        stop_arsenal_before_begin=False,
        llama_router_mode=False,
    )
except ValueError as error:
    print(f"Ожидаемая ошибка конфигурации: {error}")